In [1]:
import requests

In [2]:
HEADERS = {"User-Agent": "ai-engineering-buildcamp/1.0 (educational project)"}


In [3]:
import requests

HEADERS = {"User-Agent": "ai-engineering-buildcamp/1.0 (educational project)"}

def search_wikipedia(query: str) -> list[dict]:
    """Search Wikipedia for pages related to the query.

    Args:
        query: The search term, e.g. "capybara".

    Returns:
        A list of search results, each with 'title', 'snippet' and 'pageid'.
    """
    url = "https://en.wikipedia.org/w/api.php"
    params = {
        "action": "query",
        "format": "json",
        "list": "search",
        "srsearch": query,
    }
    response = requests.get(url, params=params, headers=HEADERS)
    response.raise_for_status()
    return response.json()["query"]["search"]

results = search_wikipedia("capybara")
print(len(results))
for r in results:
    print(r["title"])


10
Capybara
Lesser capybara
Capybara (disambiguation)
Capybara Games
Capybara Party
Capybara (software)
Hydrochoerus
Caviidae
List of largest rodents
Flow (2024 film)


In [5]:
def get_page(title: str) -> str:
    """Fetch the raw content of a Wikipedia page.

    Args:
        title: The exact title of the Wikipedia page, e.g. "Capybara".

    Returns:
        The raw wikitext of the page as a string.
    """
    url = "https://en.wikipedia.org/w/index.php"
    params = {"title": title, "action": "raw"}
    response = requests.get(url, params=params, headers=HEADERS)
    response.raise_for_status()
    return response.text

page = get_page("Capybara")
print(len(page))
print(page[:500])

44771
{{Short description|Largest species of rodents}}
{{Other uses}}
{{Good article}}
{{pp|small=yes}}
{{Use dmy dates|date=July 2022}}
{{Speciesbox
| status = LC
| status_system = IUCN3.1
| status_ref = <ref name="iucn">{{cite iucn |author=Reid, F. |year=2016 |title=''Hydrochoerus hydrochaeris'' |article-number=e.T10300A22190005 |doi=10.2305/IUCN.UK.2016-2.RLTS.T10300A22190005.en}}</ref>
| fossil_range = {{Fossilrange|Late Pleistocene|Present}}
| image = Capybaracropped.jpg
| image_alt = A capybara 


In [6]:
results = search("capybara")
print(len(results), [r["title"] for r in results])
print(len(get_page("Capybara")))

10 ['Capybara', 'Lesser capybara', 'Capybara (disambiguation)', 'Capybara Games', 'Capybara Party', 'Capybara (software)', 'Hydrochoerus', 'Caviidae', 'Flow (2024 film)', 'Below']
44771


In [4]:
results = search_wikipedia("capybara")

for r in results:
    print(r["title"])

count = sum("capybara" in r["title"].lower() for r in results)
print(count)

Capybara
Lesser capybara
Capybara (disambiguation)
Capybara Games
Capybara Party
Capybara (software)
Hydrochoerus
Caviidae
Flow (2024 film)
List of largest rodents
6


In [7]:
from pydantic_ai import Agent


instructions = """
You are a helpful assistant that answers questions using Wikipedia.

Always search Wikipedia first with search_wikipedia to find relevant pages.
Then use get_page to read the most relevant page(s) before answering.
Base your answer on the page content, and mention which pages you used.
"""

agent = Agent(
    "openai:gpt-4o-mini",
    instructions=instructions,
    tools=[search_wikipedia, get_page],
)

result = await agent.run("Where do capybaras live?")
print(result.output)

                 pydantic-ai v2.49.0 • Python 3.14.2
      / \
     /   \       agent: agent • model: openai:gpt-4o-mini • tools: 2 • capabilities: 0
   /___.___\
  /    |    \    observability: off — see every model and tool call live, with cost
/      |      \    set it up free with Logfire and a GitHub login: https://pydantic.dev/ai-setup.md
`---.._|_..---'    or use any OpenTelemetry backend: https://pydantic.dev/docs/ai/logfire/#otel

                 goes away once observability is on — or PYDANTIC_AI_NO_BANNER=1


Capybaras (Hydrochoerus hydrochaeris) are native to all countries in South America except Chile. They predominantly inhabit areas near bodies of freshwater, including rivers, lakes, ponds, streams, swamps, and reservoirs. Their preferred habitats include savannas, wetlands, and gallery forests, often flourishing in lowland regions.

These semi-aquatic rodents are social animals, typically found in groups ranging from 10 to 20 individuals, although larger groups can form. They are well-adapted to both wet and dry environments, often grazing near available water sources during the day and seeking refuge in the water during hot weather.

This information was gathered from the Wikipedia page on [Capybara](https://en.wikipedia.org/wiki/Capybara).


In [8]:
for message in result.all_messages():
    for part in message.parts:
        if part.part_kind == "tool-call":
            print(part.tool_name, part.args)

search_wikipedia {"query":"capybara habitat"}
get_page {"title":"Capybara"}


In [9]:
result = await agent.run("What are the main threats to capybara populations?")

tool_calls = []
for message in result.all_messages():
    for part in message.parts:
        if part.part_kind == "tool-call":
            tool_calls.append((part.tool_name, part.args))

print(f"Total tool calls: {len(tool_calls)}\n")
for i, (name, args) in enumerate(tool_calls, 1):
    print(f"{i}. {name} {args}")

print("\nFinal answer:\n")
print(result.output)

Total tool calls: 3

1. search_wikipedia {"query":"Capybara threats"}
2. search_wikipedia {"query":"Capybara"}
3. get_page {"title":"Capybara"}

Final answer:

The main threats to capybara populations include:

1. **Hunting**: Capybaras are hunted for their meat and hides. In some regions, they are pursued not only for food but also due to competition with livestock for grazing land. This hunting can significantly impact local populations.

2. **Habitat Loss**: The expansion of agriculture and urbanization can lead to habitat degradation and loss. Capybaras require wetland habitats, which are increasingly threatened by human activities.

3. **Predation**: They face natural predation from various South American predators, including jaguars, anacondas, ocelots, and caimans. Young or weak individuals are particularly vulnerable to these natural threats.

4. **Competition with Domestic Animals**: In some areas, capybaras are viewed as competition for grazing resources with livestock, leadi